# Posterior-BDDM CUDA Oracle Diagnostics

Runtime: choose **GPU** before running. This notebook clones the repo branch, verifies CUDA, runs the exact posterior-correction oracle diagnostic, and prints the generated report.

In [ ]:
REPO_URL = "https://github.com/Seif-Hussein/blind-diffusion-toy.git"
BRANCH = "agent/colab-cuda-oracle"  # Change to "master" after merge.

!rm -rf blind-diffusion-toy
!git clone --depth 1 --branch {BRANCH} {REPO_URL} blind-diffusion-toy
%cd blind-diffusion-toy
!python -m pip install -q -r posterior_bddm_oracle/requirements-colab.txt

In [ ]:
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device", torch.cuda.get_device_name(0))

## Smoke Test

This should finish quickly even on CPU. On GPU it verifies the same code path with CUDA tensors.

In [ ]:
!python -m posterior_bddm_oracle.src.experiments_posterior_oracle_torch \
  --quick \
  --device auto \
  --out posterior_bddm_oracle/results_cuda_smoke

## Main Posterior-Correction Oracle Sweep

The core viability metric is whether `c_split_clean` aligns with the exact posterior correction `c_star`. Increase `--n-samples`, `--components`, or the dimension list after the smoke run is stable.

In [ ]:
!python -m posterior_bddm_oracle.src.experiments_posterior_oracle_torch \
  --device auto \
  --prior ellipse \
  --d-values 2,10,50,100,500 \
  --sigmas 0.03,0.1,0.3,1.0 \
  --eta-values 1e-4,3e-4,1e-3,3e-3,1e-2,3e-2 \
  --measurement-ratios 0.5 \
  --noise-stds 0.08 \
  --n-samples 500 \
  --components 16 \
  --grid-size 61 \
  --split-methods gradient,pdhg,hqs \
  --out posterior_bddm_oracle/results_cuda

## Amplitude Sweep

Run this if the best eta in the main report is always the largest tested eta. It checks whether the split correction is only under-scaled or whether larger steps break alignment.

In [ ]:
!python -m posterior_bddm_oracle.src.experiments_posterior_oracle_torch \
  --device auto \
  --prior ellipse \
  --d-values 50,100,500 \
  --sigmas 0.3,1.0 \
  --eta-values 0.01,0.03,0.1,0.3,1.0 \
  --measurement-ratios 0.5 \
  --noise-stds 0.08 \
  --n-samples 500 \
  --components 16 \
  --grid-size 61 \
  --split-methods gradient,pdhg,hqs \
  --out posterior_bddm_oracle/results_cuda_eta_sweep

## Calibrate The Posterior Oracle

Run this before interpreting the closed-loop split hierarchy. If the posterior oracle has large covariance error, the closed-loop dynamics are not a valid sampling target yet.

In [ ]:
!python -m posterior_bddm_oracle.src.experiments_posterior_oracle_calibration_torch \
  --device auto \
  --prior ellipse \
  --d-values 100,500 \
  --h-values 0.005,0.01,0.02,0.05 \
  --beta-values 0,0.005,0.01,0.03,0.05 \
  --n-steps-values 40,80,160 \
  --measurement-ratios 0.5 \
  --noise-stds 0.08 \
  --n-trials 128 \
  --components 16 \
  --grid-size 61 \
  --out posterior_bddm_oracle/results_cuda_oracle_calibration

In [ ]:
from pathlib import Path
cal_report = Path("posterior_bddm_oracle/results_cuda_oracle_calibration/data/posterior_oracle_calibration_report.md")
print(cal_report.read_text() if cal_report.exists() else "Calibration report not found")

## Closed-Loop Blind BDDM Hierarchy

This is the first closed-loop test where blind diffusion is actually used through `sigma_hat = argmax_sigma p_sigma(Y_k)` in the candidate method.

In [ ]:
!python -m posterior_bddm_oracle.src.experiments_posterior_closed_loop_torch \
  --device auto \
  --prior ellipse \
  --d-values 50,100,500 \
  --eta-values 0.03,0.1,0.3 \
  --measurement-ratios 0.5 \
  --noise-stds 0.08 \
  --n-trials 64 \
  --n-steps 40 \
  --components 16 \
  --grid-size 61 \
  --init posterior \
  --beta 0.05 \
  --split gradient \
  --out posterior_bddm_oracle/results_cuda_closed_loop

In [ ]:
from pathlib import Path
closed_report = Path("posterior_bddm_oracle/results_cuda_closed_loop/data/closed_loop_cuda_report.md")
print(closed_report.read_text() if closed_report.exists() else "Closed-loop report not found")

In [ ]:
from pathlib import Path
report = Path("posterior_bddm_oracle/results_cuda/data/posterior_correction_cuda_report.md")
print(report.read_text() if report.exists() else "Report not found")

In [ ]:
!ls -lh posterior_bddm_oracle/results_cuda/data
!zip -qr posterior_bddm_cuda_results.zip posterior_bddm_oracle/results_cuda
print("Created posterior_bddm_cuda_results.zip")